In [2]:
import os
import math
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.parallel import DataParallel
from torchvision import transforms
from torchvision.utils import make_grid
from PIL import Image
import matplotlib.pyplot as plt

In [3]:
DATA_DIR        = "celeba_hq_256"
IMAGE_SIZE      = 32
CHANNELS        = 3

BASE_CH         = 64
TIME_DIM        = 256

BATCH_SIZE      = 256
NUM_WORKERS     = 4
LR              = 2e-4
EPOCHS          = 30
LOG_EVERY       = 50
GRAD_CLIP       = 1.0

NUM_SAMPLE_STEPS = 100
NUM_SAMPLES      = 16

OUT_DIR = "fm_celebahq"
os.makedirs(OUT_DIR, exist_ok=True)
#https://apxml.com/posts/pytorch-macos-metal-gpu
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("device:", device, "| GPUs disponibili:", torch.cuda.device_count())
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("device:", device)

device: mps


In [4]:
class CelebAHQDataset(Dataset):
    def __init__(self, root, image_size):
        root = Path(root)
        self.paths = sorted(
            list(root.rglob("*.jpg")) +
            list(root.rglob("*.jpeg")) +
            list(root.rglob("*.png"))
        )
        assert len(self.paths) > 0, f"Nessuna immagine in {root}"
        self.transform = transforms.Compose([
            transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),                       # [0, 1]
            transforms.Normalize([0.5]*3, [0.5]*3),      # [-1, 1]
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB"))

dataset = CelebAHQDataset(DATA_DIR, IMAGE_SIZE)
loader  = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)
print(f"dataset: {len(dataset)} immagini @ {IMAGE_SIZE}x{IMAGE_SIZE}")



dataset: 30000 immagini @ 32x32


In [5]:
def timestep_embedding(t, dim, max_period=10000):
    """t in [0,1] -> embedding sinusoidale di dimensione `dim`."""
    t = t * 1000.0
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(half, device=t.device) / half
    )
    args = t.float()[:, None] * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip  = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(F.silu(t_emb))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

class UNet(nn.Module):
    def __init__(self, channels=3, base=64, time_dim=256):
        super().__init__()

        # ---------- Time MLP ----------
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )

        c1, c2, c3, c4 = base, base * 2, base * 2, base * 2   # 64, 128, 128, 128

        # ---------- Stem ----------
        self.in_conv = nn.Conv2d(channels, c1, 3, padding=1)

        # ---------- Encoder (down) ----------
        self.d1a = ResBlock(c1, c1, time_dim)
        self.d1b = ResBlock(c1, c1, time_dim)
        self.down1 = nn.Conv2d(c1, c1, 4, stride=2, padding=1)   # /2

        # Livello 2
        self.d2a = ResBlock(c1, c2, time_dim)
        self.d2b = ResBlock(c2, c2, time_dim)
        self.down2 = nn.Conv2d(c2, c2, 4, stride=2, padding=1)   # /4

        # Livello 3
        self.d3a = ResBlock(c2, c3, time_dim)
        self.d3b = ResBlock(c3, c3, time_dim)
        self.down3 = nn.Conv2d(c3, c3, 4, stride=2, padding=1)   # /8

        self.d4a = ResBlock(c3, c4, time_dim)
        self.d4b = ResBlock(c4, c4, time_dim)

        # ---------- Middle ----------
        self.mid1 = ResBlock(c4, c4, time_dim)
        self.mid2 = ResBlock(c4, c4, time_dim)

        # ---------- Decoder (up) ----------
        # Livello 4 -> 3
        self.up3   = nn.ConvTranspose2d(c4, c3, 4, stride=2, padding=1)
        self.u3a   = ResBlock(c3 + c3, c3, time_dim)   # cat con skip d3b
        self.u3b   = ResBlock(c3, c3, time_dim)

        # Livello 3 -> 2
        self.up2   = nn.ConvTranspose2d(c3, c2, 4, stride=2, padding=1)
        self.u2a   = ResBlock(c2 + c2, c2, time_dim)   # cat con skip d2b
        self.u2b   = ResBlock(c2, c2, time_dim)

        # Livello 2 -> 1
        self.up1   = nn.ConvTranspose2d(c2, c1, 4, stride=2, padding=1)
        self.u1a   = ResBlock(c1 + c1, c1, time_dim)   # cat con skip d1b
        self.u1b   = ResBlock(c1, c1, time_dim)

        # ---------- Head ----------
        self.out_norm = nn.GroupNorm(8, c1)
        self.out_conv = nn.Conv2d(c1, channels, 3, padding=1)

    def forward(self, x, t):
        # tempo
        t_emb = timestep_embedding(t, self.time_dim)
        t_emb = self.time_mlp(t_emb)

        # stem
        x = self.in_conv(x)

        # ---- encoder ----
        s1 = self.d1b(self.d1a(x, t_emb), t_emb)        # skip livello 1
        x  = self.down1(s1)

        s2 = self.d2b(self.d2a(x, t_emb), t_emb)        # skip livello 2
        x  = self.down2(s2)

        s3 = self.d3b(self.d3a(x, t_emb), t_emb)        # skip livello 3
        x  = self.down3(s3)

        x  = self.d4b(self.d4a(x, t_emb), t_emb)        # bottleneck

        # ---- middle ----
        x = self.mid1(x, t_emb)
        x = self.mid2(x, t_emb)

        # ---- decoder ----
        x = self.up3(x)
        x = torch.cat([x, s3], dim=1)
        x = self.u3b(self.u3a(x, t_emb), t_emb)

        x = self.up2(x)
        x = torch.cat([x, s2], dim=1)
        x = self.u2b(self.u2a(x, t_emb), t_emb)

        x = self.up1(x)
        x = torch.cat([x, s1], dim=1)
        x = self.u1b(self.u1a(x, t_emb), t_emb)

        # ---- head ----
        return self.out_conv(F.silu(self.out_norm(x)))



In [ ]:
"""
useful sources: https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html
"""
import time
model = UNet(channels=CHANNELS, base=BASE_CH, time_dim=TIME_DIM).to(device)
print(f"parametri: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")

# if torch.cuda.device_count() > 1:
#     print(f"uso DataParallel su {torch.cuda.device_count()} GPU")
#     model = DataParallel(model)

optim  = torch.optim.AdamW(model.parameters(), lr=LR)

#scaler = torch.cuda.amp.GradScaler()

step = 0
start = time.time()
for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()

    for x1 in loader:
        x1 = x1.to(device, non_blocking=True)
        B  = x1.size(0)

        x0 = torch.randn_like(x1)
        t  = torch.rand(B, device=device)
        t_ = t.view(B, 1, 1, 1)
        xt     = (1.0 - t_) * x0 + t_ * x1
        target = x1 - x0

        # Before the backward pass, use the optimizer object to zero all of the
        # gradients for the variables it will update (which are the learnable
        # weights of the model). This is because by default, gradients are
        # accumulated in buffers( i.e, not overwritten) whenever .backward()
        # is called. Checkout docs of torch.autograd.backward for more details.
        optim.zero_grad(set_to_none=True)
        # with torch.cuda.amp.autocast(dtype=torch.float16):
        #     pred = model(xt, t)
        #     loss = F.mse_loss(pred, target)

        # Backward pass: compute gradient of the loss wrt model parameters
        loss.backward()
        # scaler.scale(loss).backward()
        # scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        # scaler.step(optim)
        # scaler.update()

        if step % LOG_EVERY == 0:
            print(f"epoch {epoch:02d} | step {step:06d} | loss {loss.item():.4f}")
        step += 1

    print(f"-> epoca {epoch} in {time.time()-t0:.1f}s")
    # salvo i pesi del modello "vero" (senza wrapper DataParallel)
    state = model.module.state_dict() if isinstance(model, DataParallel) else model.state_dict()
    torch.save(state, os.path.join(OUT_DIR, "model.pt"))

net = model.module if isinstance(model, DataParallel) else model
net.eval()
end = time.time()
print(f"tempo totale di addestramento: {end-start:.1f}s")
with torch.no_grad():
    x  = torch.randn(NUM_SAMPLES, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=device)
    dt = 1.0 / NUM_SAMPLE_STEPS
    for i in range(NUM_SAMPLE_STEPS):
        t = torch.full((NUM_SAMPLES,), i * dt, device=device)
        v = net(x, t)
        x = x + dt * v

samples = (x.clamp(-1, 1) + 1) / 2     # [-1,1] -> [0,1]
grid    = make_grid(samples.cpu(), nrow=int(math.sqrt(NUM_SAMPLES)))

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title(f"Flow Matching @ {IMAGE_SIZE}x{IMAGE_SIZE}")
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.show()